# Risk Adjustment Revenue Gap Engine
This notebook compares suspected chronic condition gaps (CDI Alerts) against RAPS and MAO-004 claims to determine which gaps are open or closed, and writes them to the Gold `memberRevenueGap` table.

In [0]:
# Setup environment and spark session
import os, sys
try:
    dbutils
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder \
        .appName("MemberRevenueGaps") \
        .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse") \
        .getOrCreate()
    from local_setup import dbutils

# Widgets/Parameters
dbutils.widgets.text("ClientContainer", "new", "Client Container")
dbutils.widgets.text("ProgramYear", "2026", "Program Year")
dbutils.widgets.text("HCCVersionRAPS", "V24", "HCC Version RAPS")
dbutils.widgets.text("HCCVersionEDPS", "V24", "HCC Version EDPS")

clientContainer = dbutils.widgets.get("ClientContainer")
programYear = int(dbutils.widgets.get("ProgramYear"))
hccVersionRAPS = dbutils.widgets.get("HCCVersionRAPS")
hccVersionEDPS = dbutils.widgets.get("HCCVersionEDPS")

print(f"Running Gap Engine for Client: {clientContainer}, Year: {programYear}")

In [0]:
# In a local test environment, if source tables don't exist, we mock them
from pyspark.sql.functions import col, lit, when

# 1. Mock HCCRaps table (Submitted RAPS Claims)
raps_data = [
    ("MEM001", "18", "2026-03-15"),  # Member 1 has diabetes claim
    ("MEM002", "19", "2026-05-10")   # Member 2 has diabetes claim
]
raps_schema = ["MemberID", "HCC", "ServiceDate"]
spark.createDataFrame(raps_data, schema=raps_schema).createOrReplaceTempView("HCCRaps")

# 2. Mock MAO004DetailDiagnosis table (Submitted Encounters)
mao_data = [
    ("MEM001", "E11.9", "18", "2026-03-15"),
    ("MEM003", "E11.9", "18", "2026-06-20")
]
mao_schema = ["MemberID", "DiagnosisCode", "HCC", "ServiceDate"]
spark.createDataFrame(mao_data, schema=mao_schema).createOrReplaceTempView("MAO004DetailDiagnosis")

# 3. Mock BasePrintedDiags table (Suspected Gaps)
# AlertResponseType: 1 = Confirmed, 2 = Rejected, null/0 = No response
alerts_data = [
    ("MEM001", "18", "HIST", 1, "2026-01-10"), # Suspected diabetes, provider confirmed, claim found
    ("MEM002", "18", "SUSP", 0, "2026-01-15"), # Suspected diabetes, no claim, open
    ("MEM003", "18", "HIST", 0, "2026-01-20"), # Suspected diabetes, claim found (MAO-004)
    ("MEM004", "19", "SUSP", 2, "2026-01-25")  # Suspected diabetes, provider rejected, closed (rejected)
]
alerts_schema = ["MemberID", "HCC", "AlertCategory", "AlertResponseType", "AlertDate"]
spark.createDataFrame(alerts_data, schema=alerts_schema).createOrReplaceTempView("BasePrintedDiags")

print("Mock source views registered.")


In [0]:
# Run comparison queries
gap_engine_query = f"""
SELECT 
     a.MemberID AS planMemberID
    ,a.HCC AS hccNumber
    ,'202601' AS reportMonth
    ,'{clientContainer}' AS clientCode
    ,cast(NULL as string) AS memberFirstName
    ,cast(NULL as string) AS memberLastName
    ,cast(NULL as string) AS memberDOB
    ,'{hccVersionRAPS}' AS hccVersion
    ,cast(NULL as string) AS hccDescription
    ,'12345' AS providerID
    ,'1234567890' AS providerNPI
    ,cast(NULL as string) AS providerLastName
    ,cast(NULL as string) AS providerFirstName
    ,cast(NULL as string) AS practiceCode
    ,cast(NULL as string) AS practiceName
    ,cast(NULL as string) AS market
    ,a.AlertCategory AS alertCategory
    ,CASE 
        WHEN r.MemberID IS NOT NULL THEN 'Closed by RAPS Claim'
        WHEN m.MemberID IS NOT NULL THEN 'Closed by MAO-004 Encounter'
        WHEN a.AlertResponseType = 1 THEN 'Closed by Provider Confirmation'
        WHEN a.AlertResponseType = 2 THEN 'Closed by Provider Rejection'
        ELSE NULL
     END AS closureReason
    ,cast(NULL as date) AS lastDCConfirmedDate
    ,cast(NULL as date) AS lastPCPVisitDate
    ,cast(NULL as date) AS lastAWVDate
    ,current_date() AS snapshotDate
    ,'PLAN001' AS planID
    ,cast(NULL as string) AS hashKey
FROM BasePrintedDiags a
LEFT JOIN HCCRaps r
    ON a.MemberID = r.MemberID AND a.HCC = r.HCC
LEFT JOIN MAO004DetailDiagnosis m
    ON a.MemberID = m.MemberID AND a.HCC = m.HCC
"""

df_gaps = spark.sql(gap_engine_query)
df_gaps.show()


In [0]:
# Write the output to Delta Silver table
spark.sql("CREATE DATABASE IF NOT EXISTS claimsprocessing.silver")
df_gaps.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("claimsprocessing.silver.silver_member_revenue_gap")
print("Revenue gaps saved successfully into claimsprocessing.silver.silver_member_revenue_gap")